# Emotion Analysis — Training Notebook
This notebook loads the labeled text data, cleans it, trains a few classifiers, compares them, and saves the best one for use in the app.

## 1. Load the data

In [ ]:
import pandas as pd

# Your working dataset: comma-separated, with a header row (columns: text, label)
df = pd.read_csv('../data/test.csv')
print(df.shape)
df.head()

In [ ]:
# Check how many examples exist per label
df['label'].value_counts().sort_index()

In [ ]:
# Peek at one example per label so we know what each number means
for label in sorted(df['label'].unique()):
    print(f"label {label}:")
    print(df[df['label'] == label]['text'].iloc[0])
    print()

## 2. Map numeric labels to emotion names
Update this if your label numbers mean something different.

In [ ]:
label_to_emotion = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

## 3. Clean the text
Lowercase, strip punctuation/numbers, remove stopwords (but keep "not", since it flips meaning).

In [ ]:
import sys
!{sys.executable} -m pip install nltk -q

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.discard('not')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

In [ ]:
sample = df['text'].iloc[0]
print("Before:", sample)
print("After: ", clean_text(sample))

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df.head()

## 4. Vectorize the text (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])

print(X.shape)

## 5. Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## 6. Train and compare a few models
`class_weight='balanced'` helps since some emotions (like "surprise") have far fewer examples than others.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

log_model = LogisticRegression(max_iter=1000, class_weight='balanced')
log_model.fit(X_train, y_train)

log_accuracy = log_model.score(X_test, y_test)
print("Logistic Regression accuracy:", log_accuracy)
print(classification_report(y_test, log_model.predict(X_test), target_names=[label_to_emotion[i] for i in sorted(label_to_emotion)]))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

dt_accuracy = dt_model.score(X_test, y_test)
print("Decision Tree accuracy:", dt_accuracy)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(class_weight='balanced')
svm_model.fit(X_train, y_train)

svm_accuracy = svm_model.score(X_test, y_test)
print("SVM accuracy:", svm_accuracy)

In [ ]:
# Quick comparison
import pandas as pd
results = pd.DataFrame({
    'model': ['Logistic Regression', 'Decision Tree', 'SVM'],
    'accuracy': [log_accuracy, dt_accuracy, svm_accuracy]
}).sort_values('accuracy', ascending=False)
results

## 7. Save the best model
Using Logistic Regression here — swap `best_model` if a different model scored higher above.

In [ ]:
import pickle
import os

best_model = log_model  # change this if another model won above

os.makedirs('../models', exist_ok=True)

with open('../models/my_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../models/my_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
with open('../models/label_mapping.pkl', 'wb') as f:
    pickle.dump(label_to_emotion, f)

print("Saved model, vectorizer, and label mapping to ../models/")

## 8. Test it by loading the saved model back

In [ ]:
import pickle

with open('../models/my_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('../models/my_vectorizer.pkl', 'rb') as f:
    loaded_vectorizer = pickle.load(f)

with open('../models/label_mapping.pkl', 'rb') as f:
    loaded_mapping = pickle.load(f)

def predict_emotion(text):
    cleaned = clean_text(text)
    vector = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vector)[0]
    return loaded_mapping[prediction]

print(predict_emotion("I am so happy today, everything is amazing"))
print(predict_emotion("I feel scared and anxious about tomorrow"))
print(predict_emotion("I can't believe this happened, I'm furious"))